In [ ]:
import meshio as mio
import h5py
import numpy as np
import pyvista as pv
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
import scipy as sp
import matplotlib.pyplot as mplt
import json as js
import meshio as mio
import subprocess as sup

# pv.set_jupyter_backend('trame')
pv.set_jupyter_backend('client')



In [ ]:
def to_pyvista_mesh(V, F = None):
    if F is None:
        return pv.PolyData(V)
    if F.shape[1] == 3:
        return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)
    elif F.shape[1] == 4:
        return pv.UnstructuredGrid({pv.CellType.TETRA: F}, V)



In [ ]:
# path = "/Users/teseo/Downloads/Embryogram test/new/0in-analysis-07_09T15_10-analysis.hdf5"
# path = "/Users/teseo/Downloads/Embryogram test/20250510_tracking-analysis-07_19T18_47-analysis.hdf5"
# path = "/Users/zoeli/Documents/UVic/masters/other/fish/still.hdf5"
path = "/Users/zoeli/Documents/UVic/masters/other/fish/newstill.hdf5"

#polyfem = "/Users/teseo/Documents/scuola/polyfem/polyfem.nosync/bin_rel.nosync/PolyFEM_bin"

In [ ]:
hdf5_file = h5py.File(path, "r")
V, T = hdf5_file["mesh/v"][:].astype(float), hdf5_file["mesh/t"][:].astype(np.int32)

top = hdf5_file["bc/top"][:].astype(np.int32)
bottom = hdf5_file["bc/bottom"][:].astype(np.int32)
middle = hdf5_file["bc/middle"][:].astype(np.int32)

In [ ]:
centers = hdf5_file["bc_func/centers"][:].astype(float)
eps = hdf5_file["bc_func/eps"][()]

In [ ]:
nk = len(hdf5_file["bc_func"].keys())-2

disps = []

for i in range(nk):
    disps.append(hdf5_file[f"bc_func/disp{i+1}"][:].astype(float))

disps[235] += np.array([0.325 * 54.1258, 0.325 * 30.7414, 0.42 * 6.51188])
disps[236] += np.array([0.325 * 54.493, 0.325 * 30.9028, 0.42 * 6.74946])
disps[237] += np.array([0.325 * 54.9033, 0.325 * 31.0096, 0.42 * 6.98708])
disps[238] += np.array([0.325 * 55.067, 0.325 * 31.1312, 0.42 * 7.02517])
disps[239] += np.array([0.325 * 55.3826, 0.325 * 31.2652, 0.42 * 6.6981])

In [ ]:
data = [np.mean(t[0]) for t in disps]
mplt.plot(data, label='x')
data = [np.mean(t[1]) for t in disps]
mplt.plot(data, label='y')
data = [np.mean(t[2]) for t in disps]
mplt.plot(data, label='z')
mplt.legend()
mplt.xlabel('time')
mplt.ylabel('disp')
mplt.title('Disp after before correction')
mplt.show()

In [ ]:
E = hdf5_file["problem/E"][()] .astype(float)
nu = hdf5_file["problem/nu"][()] .astype(float)
is_linear = hdf5_file["problem/is_linear"][()] .astype(bool)

# E

In [ ]:
# m=to_pyvista_mesh(V, T)
# # m=m.explode(0.5)
# plt = pv.Plotter()
# plt.add_mesh(m, show_edges=True, style='wireframe', line_width=1.0)
# plt.add_mesh(to_pyvista_mesh(V[top]), color='red', point_size=10, render_points_as_spheres=True, name='top')
# plt.add_mesh(to_pyvista_mesh(V[bottom]), color='green', point_size=10, render_points_as_spheres=True, name='bottom')
# plt.add_mesh(to_pyvista_mesh(V[middle]), color='blue', point_size=10, render_points_as_spheres=True, name='middle')
# plt.show()

In [ ]:
# m=m.explode(0.5)
plt = pv.Plotter()
plt.add_mesh(to_pyvista_mesh(V[middle]), color='blue', point_size=8, render_points_as_spheres=True, name='middle')
plt.add_mesh(to_pyvista_mesh(centers), color='red', point_size=7, render_points_as_spheres=True, name='middle')
plt.show()

In [ ]:
pl = pv.Plotter()
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=7, render_points_as_spheres=True, name='middle')

nd = disps[0].shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers, centers + disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='line', color='red')

def callback(x):
    vertices = np.vstack([centers, centers + disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='line', color='red')
    pl.update()

pl.show()
interact(callback, x=(0, len(disps)-1, 1))



In [ ]:
import sympy as sp 


t = sp.MatrixSymbol('t', 3, 1)

X = sp.MatrixSymbol('x', 3, 1)
M = sp.MatrixSymbol('M', 3, 3)
PP = sp.MatrixSymbol('P', 3, 3)
K = sp.MatrixSymbol('K', 3, 1)



t1=X*t.T

t2 = M*PP


t3 = M.T * K
t3[0,0]


In [ ]:
def align(centers, disps, index):
    p0 = centers.T
    p1 = (centers + disps[index]).T
    one = np.ones((1, p0.shape[1]))

    n = one.shape[1]


    X = p1 @ one.T
    PP = p1 @ p1.T
    K = p1 @ one.T
    
    system = np.zeros((9+3, 9+3))
    rhs = np.zeros((9+3, ))

    rhs[:9] = (p0 @ p1.T).flatten()
    rhs[9:] = (p0 @ one.T).flatten()


    system[0,:3] = PP[:,0]
    system[1,:3] = PP[:,1]
    system[2,:3] = PP[:,2]

    system[3,3:6] = PP[:,0]
    system[4,3:6] = PP[:,1]
    system[5,3:6] = PP[:,2]

    system[6,6:9] = PP[:,0]
    system[7,6:9] = PP[:,1]
    system[8,6:9] = PP[:,2]


    system[0, 9] = X[0,0]
    system[1, 10] = X[0,0]
    system[2, 11] = X[0,0]

    system[0, 9] = X[1,0]
    system[1, 10] = X[1,0]
    system[2, 11] = X[1,0]

    system[0, 9] = X[2,0]
    system[1, 10] = X[2,0]
    system[2, 11] = X[2,0]

    system[9,:3] = K[:,0]
    system[9,3:6] = K[:,0]
    system[9,6:9] = K[:,0]

    system[9, 9] = n
    system[10, 10] = n
    system[11, 11] = n

    sol = np.linalg.solve(system, rhs)

    t = sol[9:12]

    return sol[:9].reshape((3, 3)), np.zeros(3,) #t.reshape((3, ))

M, t = align(centers, disps, 1)

print(M)
print(t)


In [ ]:
import scipy.optimize as opt


def align_opt(centers, disps, index):
    P = centers.T
    # Q = (centers + disps[index]).T
    Q = P.copy()+ np.array([1, 2, 3]).reshape((3, 1))  # Example adjustment

    E = lambda x: np.sum((P - (x[:9].reshape(3, 3) @ Q + x[9:].reshape((3, 1))))**2)

    M = np.eye(3)
    t = np.zeros(3,)
    x0 = np.concatenate((M.flatten(), t))

    xs=opt.fmin(func=E, x0=x0, disp=True)

    # print(E(x0))
    # print(np.sum(disps[index]**2))

    Ms = xs[:9].reshape((3, 3))
    ts = xs[9:].reshape((3, ))

    return Ms, ts


align_opt(centers, disps, 1)


In [ ]:
def umeyama(P, Q):
    assert P.shape == Q.shape
    n, dim = P.shape

    centeredP = P - P.mean(axis=0)
    centeredQ = Q - Q.mean(axis=0)

    C = np.dot(np.transpose(centeredP), centeredQ) / n

    V, S, W = np.linalg.svd(C)
    d = (np.linalg.det(V) * np.linalg.det(W)) < 0.0

    if d:
        S[-1] = -S[-1]
        V[:, -1] = -V[:, -1]

    R = np.dot(V, W)

    # varP = np.var(P, axis=0).sum()
    # c = 1/varP * np.sum(S) # scale factor

    varP = np.var(P, axis=0)
    tmp = 1/varP * S # scale factor

    #ingnore z
    # tmp[2] = 1

    c = np.diag(1/tmp)

    t = Q.mean(axis=0) - P.mean(axis=0).dot(np.diag(tmp)*R)

    # use inverse
    # return c, R, -t

    #use direct
    return np.diag(tmp), R, t

def align_1(centers, disps, index):
    P = centers
    Q = (centers + disps[index])
    
    # Q = P.copy()+ np.array([1, 2, 3]).reshape((3, 1))  # Example adjustment
    # Q = 100*P+ np.array([1, 2, 3]).reshape((3, 1))
    # print(P.shape, Q.shape)

    # c, R, t = umeyama(P, Q)
    c, R, t = umeyama(Q, P)

    # return c, R, t
    # return c*R, t
    return c, t

align_1(centers, disps, 0)


In [ ]:
def align_zoe(centers, disps, index):
    P = centers
    Q = (centers + disps[index])

    assert P.shape == Q.shape
    
    centeredP = P - P.mean(axis=0)
    centeredQ = Q - Q.mean(axis=0)

    scale_x = np.linalg.lstsq(centeredP[:, 0].reshape(-1, 1), centeredQ[:, 0])[0]
    scale_y = np.linalg.lstsq(centeredP[:, 1].reshape(-1, 1), centeredQ[:, 1])[0]
    scale_z = np.linalg.lstsq(centeredP[:, 2].reshape(-1, 1), centeredQ[:, 2])[0]

    S = np.array([[scale_x[0], 0.0, 0.0],
                  [0.0, scale_y[0], 0.0],
                  [0.0, 0.0, scale_z[0]]])

    t = Q.mean(axis=0) - P.mean(axis=0).dot(S)

    return S, t

align_zoe(centers, disps, 0)

In [ ]:
def icp(centers, d):
    p0 = centers
    p1 = centers + d
    mu0 = np.mean(p0, axis=0)
    mu1 = np.mean(p1, axis=0)

    p0 -= mu0
    p1 -= mu1
    t = mu0 - mu1

    cov = p1.T @ p0
    U, s, Ut = sp.linalg.svd(cov)
    R = Ut @ U.T

    return R, t

In [ ]:
def compute_align_disp(centers, disps, index):
    # M, t = align_opt(centers, disps, index)
    # M, t = align(centers, disps, index)
    # M, t = icp(centers, disps[index])
    M, t = align_1(centers, disps, index)

    tmp = centers + disps[index]
    tmp1 = (M @ tmp.T).T + t

    return tmp1-centers, M, t

compute_align_disp(centers, disps, 1)

In [ ]:
new_disps = []
Ms = []
ts = []
for i in range(len(disps)):
    d, M, t = compute_align_disp(centers, disps, i)
    new_disps.append(d)
    Ms.append(M)
    ts.append(t)

In [ ]:
pl = pv.Plotter()
pl.add_mesh(to_pyvista_mesh(centers), color='blue', point_size=3, render_points_as_spheres=True, name='middle')

nd = disps[0].shape[0]
lines = np.hstack([[2, i, nd+i] for i in range(nd)])

vertices = np.vstack([centers, centers + disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='line', color='red')

vertices = np.vstack([centers, centers + new_disps[0]])
ll = pv.PolyData(vertices, lines=lines)
pl.add_mesh(ll, name='linea', color='green')

def callback(x):
    vertices = np.vstack([centers, centers + disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='line', color='red')

    vertices = np.vstack([centers, centers + new_disps[x]])
    ll = pv.PolyData(vertices, lines=lines)
    pl.add_mesh(ll, name='linea', color='green')
    pl.update()

    print(Ms[x])
    print(ts[x])

pl.show()
interact(callback, x=(0, len(disps)-1, 1), continuous_update=False)





In [ ]:
f=206
mplt.hist(np.linalg.norm(np.array(disps[f]-new_disps[f]), axis=1), bins=50)
mplt.show()

In [ ]:
data = [1/np.linalg.det(M) for M in Ms]
mplt.plot(data)
mplt.xlabel('time')
mplt.ylabel('scale')
mplt.show()

In [ ]:
data = [1/M[0,0] for M in Ms]
mplt.plot(data, label='x')

data = [1/M[1,1] for M in Ms]
mplt.plot(data, label='y')

data = [1/M[2,2] for M in Ms]
mplt.plot(data, label='z')

mplt.xlabel('time')
mplt.ylabel('scale')
mplt.legend()
mplt.title('Scale factors')
mplt.show()

In [ ]:
np.mean(np.linalg.norm(disps[-1][:,:2], axis=1)), np.mean(np.linalg.norm(new_disps[-1][:,:2], axis=1))

In [ ]:
disps[-1][:,:2]

In [ ]:
data = [t[0] for t in ts]
mplt.plot(data, label='x')
data = [t[1] for t in ts]
mplt.plot(data, label='y')
data = [t[2] for t in ts]
mplt.plot(data, label='z')
mplt.legend()
mplt.xlabel('time')
mplt.ylabel('translation')
mplt.show()

In [ ]:
data = [np.mean(t[0]) for t in new_disps]
mplt.plot(data, label='x')
data = [np.mean(t[1]) for t in new_disps]
mplt.plot(data, label='y')
data = [np.mean(t[2]) for t in new_disps]
mplt.plot(data, label='z')
mplt.legend()
mplt.xlabel('time')
mplt.ylabel('disp')
mplt.title('Disp after correction')
mplt.show()

In [ ]:
data = [np.mean(t[0]) for t in disps]
mplt.plot(data, label='x')
data = [np.mean(t[1]) for t in disps]
mplt.plot(data, label='y')
data = [np.mean(t[2]) for t in disps]
mplt.plot(data, label='z')
mplt.legend()
mplt.xlabel('time')
mplt.ylabel('disp')
mplt.title('Disp after before correction')
mplt.show()

In [ ]:
from scipy.interpolate import RBFInterpolator


In [ ]:
default_json = { 
"geometry": {
        "mesh": "___",
        "volume_selection": 1
    },
"boundary_conditions": {
    "dirichlet_boundary": "___"
},
"materials": {
        "E": E,
        "id": 1,
        "nu": nu,
        "type": "LinearElasticity" if is_linear else "NeoHookean"
},
"output": {
    "json": "___",
    "directory": "___",
    "paraview": {
        "file_name": "___",
        "surface": True,
        "options": {
            "material": True,
            "forces": True
        },
        "vismesh_rel_area": 10000000
    }
}
}

In [ ]:
def generate_json(out, V, T, middle, bottom, top, centers, disps, index, eps):
    rbf = RBFInterpolator(centers, disps[index])
    disp = rbf(V[middle,:])

    with open(f"{out}disp_{index}.txt", "w") as f:
        for i in range(middle.shape[0]):
            f.write(f"{middle[i]} {disp[i, 0]} {disp[i, 1]} {disp[i, 2]}\n")
        for i in range(bottom.shape[0]):
            f.write(f"{bottom[i]} 0 0 0\n")
        for i in range(top.shape[0]):
            f.write(f"{top[i]} 0 0 0\n")

    mesh = mio.Mesh(points=V, cells={"tetra": T})
    mesh.write(f"{out}mesh.msh", file_format="gmsh")

    json = default_json.copy()
    json["geometry"]["mesh"] = f"mesh.msh"
    json["boundary_conditions"]["dirichlet_boundary"] = f"disp_{index}.txt"
    json["output"]["directory"] = out
    json["output"]["json"] = f"sim{index}.json"
    json["output"]["paraview"]["file_name"] = f"sim{index}.vtu"

    with open(f"{out}run_{index}.json", "w") as f:
        js.dump(json, f, indent=4)



generate_json("outnz/", V,T, middle, bottom, top, centers, disps, 239, eps)

In [ ]:
for i in range(len(disps)):
    generate_json("outnz/", V,T, middle, bottom, top, centers, new_disps, i, eps)

In [ ]:
index=200
sup.run([polyfem, "-j", f"outnz/run_{index}.json"], check=True)